# Recreate municipality_level_analysis.csv

This notebook shows:
* how the municipality-level file was built from the Encuesta Multipropósito survey and the soil-cover data
* logic to map municipalities
* explains how the weighted rate columns are calculated.

In [2]:
import warnings
from pathlib import Path
import sys

import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')

# Resolve workspace root robustly
cwd = Path.cwd().resolve()
# If running from the src/ folder, repo root is its parent; otherwise cwd is the root
root = cwd.parent if cwd.name == 'src' else cwd
DATA_DIR = root / 'data'
MULTI_DIR = DATA_DIR / 'multiproposito'
SOIL_PATH = DATA_DIR / 'Statistics-for-Website-MB-Cobertura-col3.xlsx'

sys.path.append(str(root / 'src'))
from soil_poverty_analysis import (
    find_repo_root,
    parse_weight_series,
    normalize_municipality,
)

root = find_repo_root(root)

## 1. Load raw survey and land-cover data

We read the survey chapters that contain household-level indicators and person-level indicators, plus the land-cover workbook used to compute municipality shares.


In [3]:
def load_semicolon_csv(path, columns=None, encodings=('utf-8', 'latin1')):
    last_error = None
    for encoding in encodings:
        try:
            if columns is None:
                return pd.read_csv(path, sep=';', encoding=encoding)
            return pd.read_csv(path, sep=';', encoding=encoding, usecols=list(columns))
        except Exception as exc:
            last_error = exc
    raise last_error

ident = load_semicolon_csv(
    MULTI_DIR / 'Identificación (Capítulo A).csv',
    columns=['DIRECTORIO', 'DPTO', 'MPIO', 'NOMBRE_ESTRATO', 'FEX_C'],
    encodings=('latin1', 'utf-8'),
)
ident['weight'] = parse_weight_series(ident['FEX_C'])
ident_cund = ident[ident['DPTO'] == 25].copy()

households_c = load_semicolon_csv(
    MULTI_DIR / 'Condiciones habitacionales del hogar (Capítulo C).csv',
    columns=['DIRECTORIO', 'DIRECTORIO_HOG', 'NHCCP1', 'NHCCP31', 'NHCCP37', 'FEX_C'],
)
households_d = load_semicolon_csv(
    MULTI_DIR / 'Servicios públicos domiciliarios y de TIC (Capítulo D).csv',
    columns=['DIRECTORIO', 'DIRECTORIO_HOG', 'NHCDP1', 'NHCDP3', 'NHCDP9', 'NHCDP15', 'NHCDP28', 'FEX_C'],
)
households_l = load_semicolon_csv(
    MULTI_DIR / 'Percepción sobre las condiciones de vida y el desempeño institucional (Capítulo L).csv',
    columns=['DIRECTORIO', 'DIRECTORIO_HOG', 'NHCLP10', 'NHCLP11', 'NHCLP14', 'NHCLP16', 'NHCLP17', 'NHCLP18', 'NHCLP19', 'FEX_C'],
)
for df in [households_c, households_d, households_l]:
    df['weight'] = parse_weight_series(df['FEX_C'])

people_e = load_semicolon_csv(
    MULTI_DIR / 'Composición del hogar y demografía (Capítulo E).csv',
    columns=['DIRECTORIO', 'DIRECTORIO_HOG', 'DIRECTORIO_PER', 'SEXO', 'FEX_C'],
)
people_h = load_semicolon_csv(
    MULTI_DIR / 'Educaciвn (Capitulo H).csv',
    columns=['DIRECTORIO', 'DIRECTORIO_HOG', 'DIRECTORIO_PER', 'NPCHP24', 'NPCHP36', 'FEX_C'],
)
people_k = load_semicolon_csv(
    MULTI_DIR / 'Fuerza de trabajo (Capítulo K).csv',
    columns=['DIRECTORIO', 'DIRECTORIO_HOG', 'DIRECTORIO_PER', 'PET', 'OCU', 'DES', 'FL', 'OINFORMAL', 'FEX_C'],
)
for df in [people_e, people_h, people_k]:
    df['weight'] = parse_weight_series(df['FEX_C'])

soil = pd.read_excel(SOIL_PATH, sheet_name='COBERTURA_MUNICIPIO')
soil = soil[soil['departamento'].astype(str).str.strip().str.lower() == 'cundinamarca'].copy()
soil.head()


,municipio,departamento,pais,class_level_0,class_level_1,class_level_2,1985,1986,1987,1988,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
4492,Agua De Dios,Cundinamarca,Colombia,Antrópico,3. Área agropecuaria,3.4. Mosaico de agricultura o pasto,6986.125572,6984.431997,6974.804413,6944.673868,...,6800.615495,6759.965910,6761.391838,6783.767083,6826.467699,6796.693302,6784.748205,6775.833715,6733.846549,6712.362906
4493,Agua De Dios,Cundinamarca,Colombia,Antrópico,4. Área sin vegetación,4.2. Infraestructura urbana,20.413926,32.715782,37.262119,42.521617,...,67.838433,68.640732,69.621311,70.869324,72.295626,75.059066,76.217935,76.217935,76.574510,77.644241
4494,Agua De Dios,Cundinamarca,Colombia,Antrópico,4. Área sin vegetación,4.3. Minería,NaN,NaN,NaN,NaN,...,9.806026,9.271152,9.806029,9.627738,9.806030,9.806029,8.557987,8.290549,7.933966,6.507634
4495,Agua De Dios,Cundinamarca,Colombia,Antrópico,4. Área sin vegetación,4.5. Otra área sin vegetación,71.315001,58.834852,51.435889,47.691821,...,27.188703,25.316679,24.870945,23.088081,23.801235,23.355526,22.998938,22.642380,22.642372,21.929232
4496,Agua De Dios,Cundinamarca,Colombia,Natural,1. Formacion Boscosa,1.1. Bosque,1324.870660,1322.552874,1335.835847,1372.474058,...,1470.622139,1516.709797,1528.922664,1522.771784,1474.543986,1467.412106,1475.791321,1483.101590,1530.794587,1575.990676


## 2. Inspect input tables and match municipalities

We normalize survey municipality names and align them with the land-cover municipality names before aggregation.


In [5]:
manual_mapping = {
    'bojaca': 'Bojacá',
    'cajica': 'Cajicá',
    'chia': 'Chía',
    'facatativa': 'Facatativá',
    'fusagasuga': 'Fusagasugá',
    'gachancipa': 'Gachancipá',
    'sibate': 'Sibaté',
    'sopo': 'Sopó',
    'tocancipa': 'Tocancipá',
    'zipacon': 'Zipacón',
    'zipaquira': 'Zipaquirá',
}

ident_unique = ident_cund[['NOMBRE_ESTRATO']].drop_duplicates().copy()
ident_unique['municipio_base'] = ident_unique['NOMBRE_ESTRATO'].apply(normalize_municipality)
ident_unique['municipio_base'] = ident_unique['municipio_base'].map(lambda x: manual_mapping.get(x, x))
ident_unique['municipio_base_key'] = ident_unique['municipio_base'].apply(normalize_municipality)

soil['municipio_clean'] = soil['municipio'].astype(str).apply(normalize_municipality)
soil_lookup = soil[['municipio_clean', 'municipio']].drop_duplicates().set_index('municipio_clean')['municipio'].to_dict()
ident_unique['soil_municipio'] = ident_unique['municipio_base_key'].apply(lambda x: soil_lookup.get(x, np.nan))
ident_unique['match_method'] = np.where(
    ident_unique['municipio_base_key'].isin(soil_lookup.keys()),
    'deterministic',
    'manual',
)

match_table = ident_unique[['NOMBRE_ESTRATO', 'municipio_base', 'soil_municipio', 'match_method']].copy()
match_table.sort_values('NOMBRE_ESTRATO').head(20)


,NOMBRE_ESTRATO,municipio_base,soil_municipio,match_method
2161,Bojacá cabecera,Bojacá,Bojacá,deterministic
1945,Cajicá cabecera,Cajicá,Cajicá,deterministic
2171,Chía cabecera,Chía,Chía,deterministic
1903,Chía resto,Chía,Chía,deterministic
2414,Cota cabecera,cota,Cota,deterministic
1897,Cota resto,cota,Cota,deterministic
2139,El Rosal cabecera,el rosal,El Rosal,deterministic
2263,Facatativá cabecera,Facatativá,Facatativá,deterministic
2104,Funza cabecera,funza,Funza,deterministic
2021,Funza resto,funza,Funza,deterministic


## 3. Define weighted rate calculations

Each rate column is computed as a weighted proportion:

$$
\text{rate} = \frac{\sum_i w_i x_i}{\sum_i w_i}
$$

where $x_i$ is the indicator for household $i$ (for household-level rates) or person $i$ (for person-level rates), and $w_i$ is the survey weight.


In [6]:
def weighted_mean(series, weights):
    s = pd.to_numeric(series, errors='coerce')
    w = pd.to_numeric(weights, errors='coerce').fillna(0)
    total_w = w.sum()
    return np.average(s, weights=w) if total_w > 0 else np.nan

# Example: this is the same logic used for poor_subjective_rate and other weighted rate columns.
example = pd.Series([1, 0, 1, 0], name='indicator')
weights = pd.Series([10, 20, 30, 40], name='weight')
weighted_mean(example, weights)


np.float64(0.4)

## 4. Calculate poverty and service access rate columns

We create binary indicators from the household chapters and aggregate them to municipality level with the survey weights.


In [7]:
# Join the household chapters to the Cundinamarca identification spine
base_households = ident_cund[['DIRECTORIO', 'NOMBRE_ESTRATO']].copy()
households_c = households_c.merge(base_households, on='DIRECTORIO', how='inner')
households_d = households_d.merge(base_households, on='DIRECTORIO', how='inner')
households_l = households_l.merge(base_households, on='DIRECTORIO', how='inner')

# Prepare household indicators
households_d = households_d.rename(columns={'weight': 'weight_d'})
households_l = households_l.rename(columns={'weight': 'weight_l'})

households = households_c.merge(
    households_d[['DIRECTORIO', 'DIRECTORIO_HOG', 'NHCDP1', 'NHCDP3', 'NHCDP9', 'NHCDP15', 'NHCDP28', 'NOMBRE_ESTRATO', 'weight_d']],
    on=['DIRECTORIO', 'DIRECTORIO_HOG', 'NOMBRE_ESTRATO'],
    how='left',
)
households = households.merge(
    households_l[['DIRECTORIO', 'DIRECTORIO_HOG', 'NHCLP10', 'NHCLP11', 'NHCLP14', 'NHCLP16', 'NHCLP17', 'NHCLP18', 'NHCLP19', 'NOMBRE_ESTRATO', 'weight_l']],
    on=['DIRECTORIO', 'DIRECTORIO_HOG', 'NOMBRE_ESTRATO'],
    how='left',
)
households['weight'] = households['weight'].fillna(households['weight_d']).fillna(households['weight_l'])
households = households.drop(columns=['weight_d', 'weight_l'])

households['poor_subjective'] = (households['NHCLP11'] == 1).astype(float)
households['income_insufficient'] = (households['NHCLP10'] == 1).astype(float)
households['food_insecurity_any'] = households[['NHCLP16', 'NHCLP17', 'NHCLP18', 'NHCLP19']].eq(1).any(axis=1).astype(float)
households['tenure_own'] = (households['NHCCP1'] == 1).astype(float)
households['sanitation_access'] = (households['NHCCP31'] == 1).astype(float)
households['garbage_access'] = (households['NHCCP37'] == 1).astype(float)
households['water_access'] = (households['NHCDP1'] == 1).astype(float)
households['sewer_access'] = (households['NHCDP3'] == 1).astype(float)
households['electricity_access'] = (households['NHCDP9'] == 1).astype(float)
households['internet_access'] = (households['NHCDP28'] == 1).astype(float)

household_summary = []
for municipality, group in households.groupby('NOMBRE_ESTRATO'):
    row = {
        'NOMBRE_ESTRATO': municipality,
        'n_households': group['DIRECTORIO_HOG'].nunique(),
        'weighted_households': group['weight'].sum(),
    }
    for col in ['poor_subjective', 'income_insufficient', 'food_insecurity_any', 'tenure_own', 'sanitation_access', 'garbage_access', 'water_access', 'sewer_access', 'electricity_access', 'internet_access']:
        row[col + '_rate'] = weighted_mean(group[col], group['weight'])
    household_summary.append(row)

household_summary = pd.DataFrame(household_summary)
household_summary['municipio_base'] = household_summary['NOMBRE_ESTRATO'].apply(normalize_municipality)
household_summary['municipio_base'] = household_summary['municipio_base'].map(lambda x: manual_mapping.get(x, x))

household_summary.head()


,NOMBRE_ESTRATO,n_households,weighted_households,poor_subjective_rate,income_insufficient_rate,food_insecurity_any_rate,tenure_own_rate,sanitation_access_rate,garbage_access_rate,water_access_rate,sewer_access_rate,electricity_access_rate,internet_access_rate,municipio_base
0,Bojacá cabecera,696,3615.404305,0.326021,0.312127,0.185540,0.442731,0.991802,0.998278,0.857349,0.846927,0.863564,0.538103,Bojacá
1,Cajicá cabecera,767,23845.827506,0.141558,0.145110,0.189132,0.361345,0.998487,1.000000,0.953647,0.946714,0.953578,0.795820,Cajicá
2,Chía cabecera,832,45115.470069,0.230977,0.190935,0.207755,0.374155,0.975599,0.995919,0.906871,0.872774,0.904858,0.756842,Chía
3,Chía resto,801,8061.275261,0.355794,0.211235,0.300960,0.318220,0.703591,0.970988,0.819882,0.615723,0.855972,0.672334,Chía
4,Cota cabecera,631,8298.581479,0.220582,0.171860,0.137886,0.387412,0.942570,0.989416,0.935122,0.877765,0.946674,0.765915,cota


## 5. Compute land-cover share columns

Land-cover shares are calculated from the soil workbook by taking the area of each class within each municipality and dividing by the municipality's total area in 2021.


In [8]:
soil_long = soil[['municipio', 'departamento', 'class_level_0', 'class_level_1', 'class_level_2', 2021]].copy()
soil_long.columns = ['municipio', 'departamento', 'class_level_0', 'class_level_1', 'class_level_2', 'area_ha_2021']
soil_long['area_ha_2021'] = pd.to_numeric(soil_long['area_ha_2021'], errors='coerce').fillna(0)
soil_long['municipio_clean'] = soil_long['municipio'].astype(str).apply(normalize_municipality)
soil_long['municipio_clean'] = soil_long['municipio_clean'].map(lambda x: manual_mapping.get(x, x))

level0 = soil_long.groupby(['municipio_clean', 'class_level_0'])['area_ha_2021'].sum().reset_index()
level0['share'] = level0.groupby('municipio_clean')['area_ha_2021'].transform(lambda s: s / s.sum())
level0_wide = level0.pivot(index='municipio_clean', columns='class_level_0', values='share').reset_index()
level0_wide = level0_wide.rename(columns={'Antrópico': 'share_anthropic', 'Natural': 'share_natural'})

level1 = soil_long.groupby(['municipio_clean', 'class_level_1'])['area_ha_2021'].sum().reset_index()
level1['share'] = level1.groupby('municipio_clean')['area_ha_2021'].transform(lambda s: s / s.sum())
level1_wide = level1.pivot(index='municipio_clean', columns='class_level_1', values='share').reset_index()
level1_wide = level1_wide.rename(columns={
    '1. Formacion Boscosa': 'share_forest',
    '2. Formación natural no boscosa': 'share_natural_non_forest',
    '3. Área  agropecuaria': 'share_agriculture',
    '4. Área sin vegetación': 'share_no_vegetation',
    '5. Cuerpo de agua': 'share_water',
})

selected_classes = {
    'share_urban_infrastructure': '4.2. Infraestructura urbana',
    'share_agriculture_pasture_mosaic': '3.4. Mosaico de agricultura o pasto',
    'share_forest_level2': '1.1. Bosque',
    'share_mining': '4.3. Minería',
    'share_other_no_vegetation': '4.5. Otra área sin vegetación',
    'share_herbazales_arbustales': '2.7. Herbazales o arbustales andinos',
}
level2 = soil_long.groupby(['municipio_clean', 'class_level_2'])['area_ha_2021'].sum().reset_index()
level2['share'] = level2.groupby('municipio_clean')['area_ha_2021'].transform(lambda s: s / s.sum())
level2_wide = level2.pivot(index='municipio_clean', columns='class_level_2', values='share').reset_index()
level2_features = {feat: level2_wide[cls].fillna(0) for feat, cls in selected_classes.items()}
level2_feature_df = pd.DataFrame(level2_features, index=level2_wide['municipio_clean']).reset_index().rename(columns={'index': 'municipio_clean'})

soil_features = level0_wide.merge(level1_wide, on='municipio_clean', how='outer').merge(level2_feature_df, on='municipio_clean', how='outer')
soil_features['area_total_ha_2021'] = soil_long.groupby('municipio_clean')['area_ha_2021'].sum().reindex(soil_features['municipio_clean']).values
soil_features = soil_features.fillna(0).copy()
soil_features.head()


,municipio_clean,share_anthropic,share_natural,share_forest,share_natural_non_forest,share_agriculture,share_no_vegetation,share_water,share_urban_infrastructure,share_agriculture_pasture_mosaic,share_forest_level2,share_mining,share_other_no_vegetation,share_herbazales_arbustales,area_total_ha_2021
0,Bojacá,0.689473,0.310527,0.155476,0.154172,0.638759,0.050715,0.000879,0.0,0.0,0.0,0.0,0.0,0.0,10243.435313
1,Cajicá,0.913156,0.086844,0.071233,0.011895,0.682528,0.230628,0.003715,0.0,0.0,0.0,0.0,0.0,0.0,5106.787314
2,Chía,0.779145,0.220855,0.181549,0.037760,0.577138,0.202007,0.001546,0.0,0.0,0.0,0.0,0.0,0.0,8007.127127
3,Facatativá,0.809703,0.190297,0.156503,0.032096,0.730381,0.079323,0.001698,0.0,0.0,0.0,0.0,0.0,0.0,15797.072393
4,Fusagasugá,0.764540,0.235460,0.223671,0.011474,0.698375,0.066165,0.000314,0.0,0.0,0.0,0.0,0.0,0.0,19277.044022


## 6. Assemble municipality-level dataset

We merge the survey summaries with the land-cover shares and keep one row per municipality.


In [9]:
# Person-level indicators
people_e = people_e.merge(ident_cund[['DIRECTORIO', 'NOMBRE_ESTRATO']], on='DIRECTORIO', how='inner')
people_h = people_h.merge(ident_cund[['DIRECTORIO', 'NOMBRE_ESTRATO']], on='DIRECTORIO', how='inner')
people_k = people_k.merge(ident_cund[['DIRECTORIO', 'NOMBRE_ESTRATO']], on='DIRECTORIO', how='inner')

people_e['female'] = (people_e['SEXO'] == 2).astype(float)
people_h['scholarship'] = (people_h['NPCHP24'] == 1).astype(float)
people_h['online_courses'] = (people_h['NPCHP36'] == 1).astype(float)
people_k['pet'] = (people_k['PET'] == 1).astype(float)
people_k['ocu'] = (people_k['OCU'] == 1).astype(float)
people_k['des'] = (people_k['DES'] == 1).astype(float)
people_k['fl'] = (people_k['FL'] == 1).astype(float)
people_k['informal'] = (people_k['OINFORMAL'] == 1).astype(float)

person_summary = []
for name, df, indicator in [
    ('female', people_e, 'female'),
    ('scholarship', people_h, 'scholarship'),
    ('online_courses', people_h, 'online_courses'),
    ('pet', people_k, 'pet'),
    ('ocu', people_k, 'ocu'),
    ('des', people_k, 'des'),
    ('fl', people_k, 'fl'),
    ('informal', people_k, 'informal'),
]:
    rows = []
    for municipality, group in df.groupby('NOMBRE_ESTRATO'):
        rows.append({
            'NOMBRE_ESTRATO': municipality,
            'n_people': group['DIRECTORIO_PER'].nunique(),
            'weighted_people': group['weight'].sum(),
            f'{name}_rate': weighted_mean(group[indicator], group['weight']),
        })
    person_summary.append(pd.DataFrame(rows))

person_summary_df = person_summary[0]
for frame in person_summary[1:]:
    person_summary_df = person_summary_df.merge(frame, on=['NOMBRE_ESTRATO', 'n_people', 'weighted_people'], how='outer')

household_summary = household_summary.rename(columns={'municipio_base': 'municipio_base_hh'})
municipality_df = ident_unique[['NOMBRE_ESTRATO', 'municipio_base', 'soil_municipio', 'match_method']].merge(household_summary, on='NOMBRE_ESTRATO', how='left')
municipality_df = municipality_df.merge(person_summary_df, on='NOMBRE_ESTRATO', how='left')
soil_features_renamed = soil_features.rename(columns={'municipio_clean': 'municipio_base'})
municipality_df = municipality_df.merge(soil_features_renamed, on='municipio_base', how='left')
municipality_df = municipality_df[municipality_df['municipio_base'].notna()].copy()

# Aggregate any duplicate municipality rows created by cabecera/resto splits.
def aggregate_by_municipality(df):
    group_cols = ['municipio_base']
    count_cols = ['n_households', 'weighted_households', 'n_people', 'weighted_people']
    rate_cols = [c for c in df.columns if c.endswith('_rate') and c not in count_cols]
    constant_cols = [c for c in df.columns if c not in group_cols + count_cols + rate_cols + ['NOMBRE_ESTRATO', 'soil_municipio', 'match_method', 'municipio_base_key']]
    out = []
    for municipality, group in df.groupby(group_cols, dropna=False):
        municipality_value = municipality[0] if isinstance(municipality, tuple) else municipality
        row = {'municipio_base': municipality_value}
        for c in count_cols:
            row[c] = group[c].sum() if c in group.columns else np.nan
        for c in rate_cols:
            if 'weighted_households' in group.columns and c.startswith(('poor_', 'income_', 'food_', 'tenure_', 'sanitation_', 'garbage_', 'water_', 'sewer_', 'electricity_', 'internet_')):
                weights = group['weighted_households']
            elif 'weighted_people' in group.columns and c.endswith('_rate'):
                weights = group['weighted_people']
            else:
                weights = pd.Series(1, index=group.index)
            row[c] = weighted_mean(group[c], weights)
        for c in constant_cols:
            row[c] = group[c].iloc[0] if c in group.columns else np.nan
        out.append(row)
    return pd.DataFrame(out)

municipality_df = aggregate_by_municipality(municipality_df)
municipality_df = municipality_df.sort_values('municipio_base').reset_index(drop=True)
municipality_df.head()


,municipio_base,n_households,weighted_households,n_people,weighted_people,poor_subjective_rate,income_insufficient_rate,food_insecurity_any_rate,tenure_own_rate,sanitation_access_rate,...,share_agriculture,share_no_vegetation,share_water,share_urban_infrastructure,share_agriculture_pasture_mosaic,share_forest_level2,share_mining,share_other_no_vegetation,share_herbazales_arbustales,area_total_ha_2021
0,Bojacá,2088,10846.212914,4932,26749.390163,0.326021,0.312127,0.185540,0.442731,0.991802,...,0.638759,0.050715,0.000879,0.0,0.0,0.0,0.0,0.0,0.0,10243.435313
1,Cajicá,2301,71537.482519,6106,193444.552575,0.141558,0.145110,0.189132,0.361345,0.998487,...,0.682528,0.230628,0.003715,0.0,0.0,0.0,0.0,0.0,0.0,5106.787314
2,Chía,4899,159530.235991,13721,445597.990538,0.249899,0.194012,0.221885,0.365675,0.934364,...,0.577138,0.202007,0.001546,0.0,0.0,0.0,0.0,0.0,0.0,8007.127127
3,Facatativá,2463,147326.276894,7090,430240.322703,0.370756,0.293271,0.375271,0.288027,0.987942,...,0.730381,0.079323,0.001698,0.0,0.0,0.0,0.0,0.0,0.0,15797.072393
4,Fusagasugá,2847,169259.256946,6466,397623.128564,0.315716,0.261989,0.357895,0.335408,0.967199,...,0.698375,0.066165,0.000314,0.0,0.0,0.0,0.0,0.0,0.0,19277.044022
